# Phase 3 — Transformer Fine-Tuning on Colab T4 (thin wrapper)

This notebook is a **thin driver** — it contains **no model or training logic**. All of it
lives in `src/models/transformer_train.py` (`train_stage`, `export_backbone`,
`WeightedTrainer`) and the 03-02 leaves it composes. The notebook only: installs the pinned
stack, mounts Drive, clones the repo, and calls those functions for each (backbone, stage).

**Why Colab (D-05):** fine-tuning is GPU-bound; the local Mac Mini M4 has no CUDA. We train
on a free **T4** here, `save_pretrained` the cascade to mounted Drive, then download
`models/transformer/` into the local repo for **CPU inference** (the local loader imports
**zero** training code).

**Both backbones end-to-end (D-10):** `csebuetnlp/banglishbert` (code-mixed primary) and
`xlm-roberta-base` (robust all-rounder) are each fine-tuned for both cascade heads
(gate → real/fake). Per-language macro-F1 selection happens in 03-04, not here.

**Session loss (Pitfall 4):** `TRAIN_HPARAMS` uses `save_strategy="epoch"`, short 2-3 epoch
fine-tunes, fp16, batch 8-16 — so a dropped Colab session resumes from the last epoch
checkpoint on Drive.

## 1. Install the pinned stack

Matches the CLAUDE.md / 03-RESEARCH Standard Stack. `csebuetnlp/normalizer` is **git-only**
and **mandatory** before BanglaBERT/BanglishBERT tokenization (pretraining-inference parity,
D-12 — skipping it silently degrades accuracy). Colab is a fresh env, so the numpy-2 caveat
from the local `.venv` (A1) usually does not apply here; pin only if an import error appears.

In [ ]:
!pip install -q \
    'torch==2.6.*' 'transformers==4.46.*' 'accelerate>=0.26' \
    datasets evaluate sentencepiece pandas pyarrow
!pip install -q 'git+https://github.com/csebuetnlp/normalizer'

# Colab preinstalls torchvision/torchaudio built for a NEWER torch; under our pinned torch
# 2.6 their compiled ops (e.g. torchvision::nms) are ABI-mismatched, and transformers 4.46
# imports torchvision at model-load time -> "operator torchvision::nms does not exist". We
# use neither, so remove them and transformers skips the import. RESTART RUNTIME after this cell.
!pip uninstall -y -q torchvision torchaudio

## 2. Mount Drive + clone/pull the repo

Drive holds the checkpoints (epoch saves) and the final exported `models/transformer/`. Set
`REPO_URL` to your remote; the processed parquet splits must be present under
`data/processed/` (they are gitignored — sync them to the repo dir on Colab or to Drive).

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

REPO_URL = 'https://github.com/HasanJahidul/fake-news-detection.git'  # <-- set to your remote
REPO_DIR = '/content/fake-news-detection'
DRIVE_OUT = '/content/drive/MyDrive/fnd_models/transformer'  # exported cascade lands here

if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull

%cd $REPO_DIR
os.makedirs(DRIVE_OUT, exist_ok=True)

## 3. Fine-tune both backbones × both heads, fit val temperature, export

Pure composition over `transformer_train`: for each backbone we `train_stage` the gate, fit
its val temperature, `train_stage` the real/fake head, fit its val temperature, then
`export_backbone` the locked layout (`gate/`, `realfake/`, `label_map.json` from `LABELS`,
`temperature.json`) to Drive. The gate threshold is left at the 0.5 sentinel — 03-04 sweeps
and records the chosen val threshold.

In [ ]:
from pathlib import Path

from src.models.calibration import fit_temperature
from src.models.transformer_data import load_splits
from src.models.transformer_train import (
    BACKBONES,
    STAGE_GATE,
    STAGE_REALFAKE,
    export_backbone,
    train_stage,
)

train_df, val_df, _test_df = load_splits()

for backbone_key, model_id in BACKBONES.items():
    print(f'=== fine-tuning {backbone_key} ({model_id}) ===')

    gate_model, gate_tok, gate_logits, gate_y = train_stage(
        model_id, STAGE_GATE, train_df, val_df
    )
    gate_T = fit_temperature(gate_logits, gate_y)

    rf_model, rf_tok, rf_logits, rf_y = train_stage(
        model_id, STAGE_REALFAKE, train_df, val_df
    )
    rf_T = fit_temperature(rf_logits, rf_y)

    out_dir = Path(DRIVE_OUT) / backbone_key
    export_backbone(
        backbone_key,
        gate_model, gate_tok,
        rf_model, rf_tok,
        gate_T=gate_T, realfake_T=rf_T,
        gate_threshold=0.5,
        out_dir=out_dir,
    )
    print(f'exported -> {out_dir}  (gate_T={gate_T:.3f}, rf_T={rf_T:.3f})')

## 4. Download `models/transformer/` for local inference (D-05)

The exported cascade now lives on Drive at `MyDrive/fnd_models/transformer/<backbone>/`.
Copy that whole `transformer/` folder into your **local** repo at
`models/transformer/` (the dir is gitignored — the safetensors artifacts are not committed,
they are rebuilt by this notebook). Local CPU inference (03-04 loader) then runs
`preprocess() → tokenize → gate → (if not malicious) real/fake → calibrated path-product
confidence` with **no training-code import**.

```bash
# on the local machine, after downloading transformer/ from Drive:
#   mv ~/Downloads/transformer  <repo>/models/transformer
# then: python -m src.models.select_transformer   # 03-04 picks the winner
```